In [ ]:
import numpy as np
import torch

from pid.models import CartesianProductFlow, BroadcastChannelFlow
from pid.optimizers import exact_gauss_tilde_pid, exact_gauss_thin_pid, flow_pid
import pid.models.lmi as lmi
from pid.utils import generate_randomBC

from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
dm = 2  # Number of dimensions of M
dx = 512
dy = 512

# set manual seed
torch.manual_seed(10)
np.random.seed(10)

hx, hy = generate_randomBC(dm, dx, dy)

sigm = np.eye(dm)
sigx_m = np.eye(dx)
sigy_m = np.eye(dy)
sigw = np.zeros((dx, dy))

# Covariance matrix construction for both unique or redundant
cov = np.block([[sigm, sigm @ hx.T, sigm @ hy.T],
                [hx @ sigm, hx @ sigm @ hx.T + sigx_m, hx @ sigm @ hy.T + sigw],
                [hy @ sigm, hy @ sigm @ hx.T + sigw.T, hy @ sigm @ hy.T + sigy_m]])

print(cov.shape)

In [ ]:
n = 10000   # number of samples

mean = np.zeros(dm+dx+dy)
mxy = np.random.multivariate_normal(mean, cov, size=n)
cov = np.corrcoef(mxy.T)

mean = np.zeros(dm+dx+dy)
mxy = np.random.multivariate_normal(mean, cov, size=n)
m_data, x_data, y_data = np.hsplit(mxy, [dm, dm+dx])  # split m, x, y
print(m_data.shape, x_data.shape, y_data.shape)

x_data = x_data**3
y_data = np.cbrt(y_data)
m_data = np.cbrt(m_data)

def standardize_data(data):
    data = torch.from_numpy(np.nan_to_num((data - data.mean(axis=0)) / data.std(axis=0))).float()
    data = torch.clip(data, min=-10, max=10)
    return data

m_standardized = standardize_data(m_data)
x_standardized = standardize_data(x_data)
y_standardized = standardize_data(y_data)

dataset = TensorDataset(x_standardized, y_standardized, m_standardized)
train_loader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

In [ ]:
new_mxy = np.hstack((m_data, x_data, y_data))
new_cov = np.corrcoef(new_mxy.T)
print(f"New covariance shape: {new_cov.shape}")

In [ ]:
enable_cuda=True
device = torch.device('cuda' if torch.cuda.is_available() and enable_cuda else 'cpu')
print(f"Using device: {device}")

latent_dim = 8

lmi_encoder = lmi.AEMINE(dx, dy, dm, latent_dim, alpha=1.0, lam=1.0)
model = CartesianProductFlow(
    dm=dm,
    dx=dx,
    dy=dy,
    n_flows=6,
).to(device)

In [ ]:
# Train model
max_iter = 5000
train_iter = iter(train_loader)

loss_hist = np.array([])

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iter, eta_min=1e-5)

for i in tqdm(range(max_iter)):
    try:
        x, y, m = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, y, m = next(train_iter)
    optimizer.zero_grad()
    loss = model.learning_loss(m.to(device), x.to(device), y.to(device))

    if ~(torch.isnan(loss) | torch.isinf(loss)):
        loss.backward()
        optimizer.step()
        scheduler.step()

    loss_hist = np.append(loss_hist, loss.detach().to('cpu').numpy())

print(f"Final loss: {loss_hist[-1]}")

In [ ]:
plt.figure(figsize=(3, 3))
plt.plot(loss_hist, label='loss')
plt.legend()
plt.show()

In [ ]:
ret = exact_gauss_tilde_pid(cov, dm, dx, dy)
imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
print(f"Truth: R: {ri}, U1: {uix}, U2: {uiy}, S: {si}")

ret = exact_gauss_tilde_pid(new_cov, dm, dx, dy)
imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
print(f"Tilde-PID: R: {ri}, U1: {uix}, U2: {uiy}, S: {si}")

# pid_flow = flow_pid.fit_flows(model, dataloader, n_epochs=50, lr=2e-4, device=device, verbose=True)
ret = flow_pid.fit_pid(model, train_loader, device=device, verbose=True)
imxy, uix, uiy, ri, si = ret[2], *ret[-4:]
print(f"Flow pid, R: {ret[7]}, U1: {ret[5]}, U2: {ret[6]}, S: {ret[8]}")

In [ ]:
m_list = []
x_list = []
y_list = []
for x_batch, y_batch, m_batch in train_loader:
    x_batch, y_batch, m_batch = x_batch.to(device), y_batch.to(device), m_batch.to(device)
    with torch.no_grad():
        z_m, z_x, z_y = model(m_batch, x_batch, y_batch)
    m_list.append(z_m)
    x_list.append(z_x)
    y_list.append(z_y)
z_x = torch.cat(x_list, dim=0).cpu().numpy()
z_y = torch.cat(y_list, dim=0).cpu().numpy()
z_m = torch.cat(m_list, dim=0).cpu().numpy()

print(f"z_m shape: {z_m.shape}, z_x shape: {z_x.shape}, z_y shape: {z_y.shape}")

In [ ]:
from sklearn.decomposition import PCA

n_comps = 256  # Number of components to keep
n_comps = min(dx, dy, n_comps)

pca_x = PCA(n_components=n_comps)
pca_y = PCA(n_components=n_comps)
x_hat = pca_x.fit_transform(z_x)
y_hat = pca_y.fit_transform(z_y)

mxy_hat = np.hstack((z_m, x_hat, y_hat))
cov_hat = np.cov(mxy_hat.T)

ret = exact_gauss_thin_pid(cov_hat, dm, n_comps, n_comps)
print(f"Thin-PID: R: {ret[7]}, U1: {ret[5]}, U2: {ret[6]}, S: {ret[8]}")